# Lab 8: SQL Fundamentals With a Mini Business Database

In this lab, you will practice the core SQL topics from the Week 3 review using a small in-memory database.

This notebook uses Python's built-in `sqlite3` module so you can run the lab without installing Postgres. Most query concepts transfer directly to Postgres, but a few syntax details differ. Notes are included where that matters.

## Objectives

By completing this lab, you will be able to:

- Define tables with primary keys, foreign keys, `NOT NULL`, `UNIQUE`, and `CHECK` constraints.
- Insert, read, update, and delete rows with DML statements.
- Query data with filtering, sorting, grouping, aggregate functions, and scalar functions.
- Use `INNER JOIN`, `LEFT JOIN`, self joins, and cross joins.
- Write subqueries and set operations.
- Create views and indexes.
- Explain relationship cardinality, normalization, and transaction behavior.

## Setup

Run the next cell first. It creates a temporary database in memory. If you restart the kernel, run this setup cell again.

In [1]:
import sqlite3
from pprint import pprint

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
connection.execute("PRAGMA foreign_keys = ON;")

def run_query(sql, params=None):
    """Run a SELECT query and return rows as dictionaries."""
    cursor = connection.execute(sql, params or [])
    rows = [dict(row) for row in cursor.fetchall()]
    for row in rows:
        print(row)
    return rows

def run_statement(sql, params=None):
    """Run an INSERT, UPDATE, DELETE, or DDL statement."""
    connection.execute(sql, params or [])
    connection.commit()
    print("Statement completed.")

## Part 1: DDL and Constraints

The next cell creates a small business database with customers, products, orders, order items, support tickets, and employees.

Look for examples of:

- `PRIMARY KEY`
- `FOREIGN KEY`
- `NOT NULL`
- `UNIQUE`
- `CHECK`

Postgres note: `INTEGER PRIMARY KEY AUTOINCREMENT` in SQLite is similar in spirit to `SERIAL PRIMARY KEY` or `GENERATED ... AS IDENTITY` in Postgres.

In [2]:
schema_sql = """
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    country TEXT NOT NULL,
    signup_date TEXT NOT NULL,
    is_active INTEGER NOT NULL CHECK (is_active IN (0, 1))
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL UNIQUE,
    category TEXT NOT NULL,
    price NUMERIC NOT NULL CHECK (price >= 0)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('pending', 'paid', 'shipped', 'cancelled')),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price NUMERIC NOT NULL CHECK (unit_price >= 0),
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE support_tickets (
    ticket_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    subject TEXT NOT NULL,
    priority TEXT NOT NULL CHECK (priority IN ('low', 'medium', 'high')),
    opened_at TEXT NOT NULL,
    resolved_at TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_name TEXT NOT NULL,
    manager_id INTEGER,
    FOREIGN KEY (manager_id) REFERENCES employees(employee_id)
);
"""

connection.executescript(schema_sql)
print("Schema created.")

Schema created.


### Exercise 1

Answer in a markdown cell below:

1. Which table represents a many-to-many relationship?
2. Which tables demonstrate a one-to-many relationship?
3. Which table can be used for a self join?

Your answer:

- Many-to-many:
- One-to-many:
- Self join:

## Part 2: DML - Insert Sample Data

Run the next cell to populate the database.

In [4]:
seed_sql = """
INSERT INTO customers (first_name, last_name, email, country, signup_date, is_active) VALUES
('Alice', 'Smith', 'alice@example.com', 'USA', '2024-01-10', 1),
('Bob', 'Johnson', 'bob@example.com', 'Canada', '2024-01-15', 1),
('Carol', 'Nguyen', 'carol@example.com', 'USA', '2024-02-20', 0),
('David', 'Lopez', 'david@example.com', 'Mexico', '2024-03-05', 1),
('Eva', 'Patel', 'eva@example.com', 'India', '2024-04-12', 1);

INSERT INTO products (product_name, category, price) VALUES
('Analytics Starter', 'Software', 49.00),
('Data Warehouse Pro', 'Software', 299.00),
('SQL Bootcamp', 'Training', 799.00),
('Dashboard Template', 'Template', 99.00),
('Support Plan', 'Service', 149.00);

INSERT INTO orders (customer_id, order_date, status) VALUES
(1, '2024-02-01', 'paid'),
(1, '2024-03-15', 'shipped'),
(2, '2024-03-20', 'paid'),
(3, '2024-04-01', 'cancelled'),
(4, '2024-04-18', 'pending');

INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES
(1, 1, 2, 49.00),
(1, 4, 1, 99.00),
(2, 2, 1, 299.00),
(3, 3, 1, 799.00),
(4, 1, 1, 49.00),
(5, 5, 2, 149.00);

INSERT INTO support_tickets (customer_id, subject, priority, opened_at, resolved_at) VALUES
(1, 'Invoice question', 'low', '2024-02-02', '2024-02-03'),
(1, 'Dashboard export failed', 'high', '2024-03-17', NULL),
(2, 'Password reset', 'medium', '2024-03-21', '2024-03-21'),
(5, 'Pricing question', 'low', '2024-04-20', NULL);

INSERT INTO employees (employee_name, manager_id) VALUES
('Maya Director', NULL),
('Noah Manager', 1),
('Lina Analyst', 2),
('Omar Engineer', 2);
"""

connection.executescript(seed_sql)
connection.commit()
print("Sample data inserted.")

Sample data inserted.


## Part 3: SELECT, Filtering, Ordering, and Scalar Functions

Complete each TODO query. Use `run_query(...)` to see results.

In [5]:
# Example: show all active customers sorted by signup date.
run_query("""
SELECT customer_id, first_name, last_name, country, signup_date
FROM customers
WHERE is_active = 1
ORDER BY signup_date;
""")

{'customer_id': 1, 'first_name': 'Alice', 'last_name': 'Smith', 'country': 'USA', 'signup_date': '2024-01-10'}
{'customer_id': 2, 'first_name': 'Bob', 'last_name': 'Johnson', 'country': 'Canada', 'signup_date': '2024-01-15'}
{'customer_id': 4, 'first_name': 'David', 'last_name': 'Lopez', 'country': 'Mexico', 'signup_date': '2024-03-05'}
{'customer_id': 5, 'first_name': 'Eva', 'last_name': 'Patel', 'country': 'India', 'signup_date': '2024-04-12'}


[{'customer_id': 1,
  'first_name': 'Alice',
  'last_name': 'Smith',
  'country': 'USA',
  'signup_date': '2024-01-10'},
 {'customer_id': 2,
  'first_name': 'Bob',
  'last_name': 'Johnson',
  'country': 'Canada',
  'signup_date': '2024-01-15'},
 {'customer_id': 4,
  'first_name': 'David',
  'last_name': 'Lopez',
  'country': 'Mexico',
  'signup_date': '2024-03-05'},
 {'customer_id': 5,
  'first_name': 'Eva',
  'last_name': 'Patel',
  'country': 'India',
  'signup_date': '2024-04-12'}]

In [ ]:
# TODO 1: Select each customer's full name and uppercase country.
# Hint: SQLite uses || for string concatenation and UPPER(...) for uppercase.
run_query("""
SELECT
    -- replace this line
    first_name
FROM customers;
""")

In [ ]:
# TODO 2: Find products priced between 50 and 300, ordered from most expensive to least expensive.
run_query("""
SELECT product_name, category, price
FROM products
-- WHERE ...
-- ORDER BY ...
""")

## Part 4: Aggregates, GROUP BY, and HAVING

Aggregate functions collapse many rows into a summary value. Common examples are `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`.

In [ ]:
# Example: count customers by country.
run_query("""
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country
ORDER BY customer_count DESC;
""")

In [ ]:
# TODO 3: Calculate total revenue per order.
# Revenue per item is quantity * unit_price.
run_query("""
SELECT
    order_id
    -- add total_revenue
FROM order_items
GROUP BY order_id;
""")

In [ ]:
# TODO 4: Find categories with average product price greater than 100.
# Hint: use GROUP BY and HAVING.
run_query("""
SELECT category, AVG(price) AS avg_price
FROM products
-- GROUP BY ...
-- HAVING ...
""")

## Part 5: Joins

Use joins to combine rows from related tables. Most joins in application databases connect a foreign key to a primary key.

In [ ]:
# Example: orders with customer names.
run_query("""
SELECT
    o.order_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    o.order_date,
    o.status
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")

In [ ]:
# TODO 5: Show every customer and any support tickets they have.
# Include customers even when they have no tickets.
# Hint: LEFT JOIN.
run_query("""
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name
    -- add ticket fields
FROM customers AS c
-- LEFT JOIN ...
ORDER BY c.customer_id;
""")

In [ ]:
# TODO 6: Use a self join to show each employee and their manager name.
run_query("""
SELECT
    e.employee_name AS employee
    -- add manager name
FROM employees AS e
-- LEFT JOIN employees AS m ON ...
""")

## Part 6: Subqueries and Set Operations

Subqueries let one query use the result of another query. Set operations combine result sets.

In [ ]:
# TODO 7: Find products whose price is above the average product price.
run_query("""
SELECT product_name, price
FROM products
WHERE price > (
    -- write a subquery that returns the average price
    SELECT 0
)
ORDER BY price DESC;
""")

In [ ]:
# Example: UNION creates one stacked result from compatible SELECT statements.
run_query("""
SELECT email AS contact, 'customer' AS contact_type FROM customers
UNION
SELECT subject AS contact, 'ticket subject' AS contact_type FROM support_tickets;
""")

## Part 7: Views and Indexes

A view is a saved query. An index can speed up reads at the cost of extra storage and write overhead.

In [ ]:
connection.executescript("""
CREATE VIEW order_revenue AS
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.status,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders AS o
JOIN order_items AS oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.customer_id, o.order_date, o.status;

CREATE INDEX idx_orders_customer_id ON orders(customer_id);
""")
connection.commit()

run_query("SELECT * FROM order_revenue ORDER BY revenue DESC;")

### Exercise 2

Write a query against the `order_revenue` view that returns only paid or shipped orders with revenue greater than 100.

In [ ]:
# TODO 8: Query the order_revenue view.
run_query("""
SELECT *
FROM order_revenue
-- WHERE ...
""")

## Part 8: Transactions and ACID

A transaction groups changes so they succeed or fail together.

- Atomicity: all or nothing
- Consistency: constraints and relationships stay valid
- Isolation: transactions do not interfere unexpectedly
- Durability: committed changes persist

The next cell intentionally rolls back a change.

In [ ]:
print("Before transaction:")
run_query("SELECT product_id, product_name, price FROM products WHERE product_id = 1;")

connection.execute("BEGIN;")
connection.execute("UPDATE products SET price = 10 WHERE product_id = 1;")
print("Inside transaction, before rollback:")
run_query("SELECT product_id, product_name, price FROM products WHERE product_id = 1;")

connection.rollback()
print("After rollback:")
run_query("SELECT product_id, product_name, price FROM products WHERE product_id = 1;")

## Part 9: Normalization and ERD Thinking

Answer these questions in the markdown cell below:

1. Why does `order_items` use a composite primary key?
2. Why is product price stored in `products`, but `unit_price` also stored in `order_items`?
3. Which columns would be repeated too much if this database used one giant table?
4. Identify one table that is mostly 3NF and explain why.
5. Sketch the ERD on paper or with a diagram tool. Label 1:1, 1:n, and n:n relationships where they apply.

Your answers:

1.
2.
3.
4.
5.

## Stretch Goals

- Add a `payments` table and decide whether it is 1:1 or 1:n with `orders`.
- Add a `refunds` table and write a query for net revenue.
- Create a view for customer lifetime value.
- Try to insert invalid data that violates a constraint, then explain the error.
- Rewrite one query using a subquery and again using a join. Which is easier to read?
- If using Postgres later, repeat this lab with `psycopg2` or `psql` and compare syntax differences.

## Reflection

1. What is the difference between DDL and DML?
2. Why do foreign keys matter for data integrity?
3. When would you use `HAVING` instead of `WHERE`?
4. What does a `LEFT JOIN` show that an `INNER JOIN` might hide?
5. Why can indexes improve reads but slow down writes?